# Laboratorio 2 - Simulación Computacional
## Secciones 5.2, 5.3 y 5.4

**Universidad de los Llanos - Escuela de Ingeniería**
**Curso:** Simulación Computacional
**Práctica Nº 02:** Números (pseudo)aleatorios y Generación de Variables Aleatorias

**Docente:** Angel Alfonso Cruz Roa

**Estudiante:** Luciana A. Belandria A.

**Código:** 160005005


# 0. Librerías y dependencias

In [ ]:
import numpy as np
import math
import time
import matplotlib.pyplot as plt
from scipy import stats

np.set_printoptions(precision=4, suppress=True)

Generador congruencial mixto

In [ ]:
def genran(a, c, m, xant):
    xsig = (a * xant + c) % m
    usig = xsig / m
    return [xsig, usig]

def genranN(a, c, m, x0, N):
    xant = x0
    I = []
    X = []
    U = []
    for t in range(1, N + 1):
        [xi, ui] = genran(a, c, m, xant)
        I.append(t)
        X.append(xi)
        U.append(ui)
        xant = xi
    return [I, X, U]

def showValues(I, X, U):
    print("i\tXi\tUi")
    for t in range(0, len(X)):
        print("%d\t%d\t%2.4f" % (I[t], X[t], U[t]), sep=' ', end='\n')

# 5.2. Uso de números aleatorios para evaluar integrales

Siguiendo la sección 3.2 de [Ross99], para estimar $\theta$ se reescribe la integral como un valor
esperado respecto a variables $U(0,1)$ y se aproxima por el promedio muestral (Monte Carlo)
Para $(-\infty,\infty)$ se aprovecha la simetría o se usa el cambio en las dos variables.

## Código base Integración Monte Carlo

In [ ]:
def calcular_integral(N):
    start_time = time.time()
    np.random.seed(0)

    n = 0
    suma = 0
    while ( n < N ):
        x = 2 * np.random.rand()
        y = 2 * np.exp(-x*x)
        suma = suma + y
        n = n + 1
    print('n: %d \t Resultado: %6.6f \t %6.6f segundos'% (N ,  (suma/n),(time.time() - start_time)))

calcular_integral(10)
calcular_integral(100)
calcular_integral(1000)
calcular_integral(10000)
calcular_integral(100000)
calcular_integral(1000000)

n: 10 	 Resultado: 0.546039 	 0.000331 segundos
n: 100 	 Resultado: 0.940202 	 0.000271 segundos
n: 1000 	 Resultado: 0.897256 	 0.002620 segundos
n: 10000 	 Resultado: 0.890560 	 0.027435 segundos
n: 100000 	 Resultado: 0.883501 	 0.253660 segundos
n: 1000000 	 Resultado: 0.881186 	 2.575902 segundos


## Método general de Monte Carlo

Se generaliza el código anterior a los tres casos y se usa el generador congruencial mixto como
fuente de los $u_i$. `montecarlo` devuelve la estimación y el error estándar.

In [ ]:
A_MC, C_MC, M_MC = 1664525, 1013904223, 2**32

def montecarlo(g, N, a=0.0, b=1.0, tipo='ab', x0=12345):

    _, _, U = genranN(A_MC, C_MC, M_MC, x0, N)
    U = np.array(U)
    if tipo == '01':
        muestras = g(U)
    elif tipo == 'ab':
        muestras = (b - a) * g(a + (b - a) * U)
    elif tipo == 'inf':
        muestras = g(1.0/U - 1.0) / (U**2)
    elif tipo == 'inf_sim':
        muestras = 2.0 * g(1.0/U - 1.0) / (U**2)
    else:
        raise ValueError("tipo no válido")
    est = np.mean(muestras)
    err = np.std(muestras, ddof=1) / np.sqrt(N)
    return est, err

### Ejercicios 3 a 7


In [ ]:
integrales = [
    ("3. ∫_0^1 e^{e^x} dx",          lambda x: np.exp(np.exp(x)),       '01',     0.0, 1.0, 6.316564),
    ("4. ∫_0^1 (1-x^2)^{3/2} dx",    lambda x: np.power(1-x**2, 1.5),   '01',     0.0, 1.0, 3*np.pi/16),
    ("5. ∫_{-2}^2 e^{x+x^2} dx",     lambda x: np.exp(x + x**2),        'ab',    -2.0, 2.0, 93.162753),
    ("6. ∫_0^inf x(1+x^2)^{-2} dx",  lambda x: x*np.power(1+x**2, -2.0),'inf',    0.0, 1.0, 0.5),
    ("7. ∫_{-inf}^{inf} e^{-x^2} dx",lambda x: np.exp(-x*x),            'inf_sim',0.0, 1.0, np.sqrt(np.pi)),
]

Ns = [100, 1000, 10000, 100000, 1000000]

for nombre, g, tipo, a, b, exacto in integrales:
    print("=" * 78)
    print(nombre, "   (valor exacto = %.6f)" % exacto)
    print("=" * 78)
    print("%-12s %-14s %-14s %-12s" % ("N", "Estimado", "Error abs.", "Error est."))
    for N in Ns:
        est, err = montecarlo(g, N, a=a, b=b, tipo=tipo, x0=12345)
        print("%-12d %-14.6f %-14.6f %-12.6f" % (N, est, abs(est-exacto), err))
    print()

3. ∫_0^1 e^{e^x} dx    (valor exacto = 6.316564)
N            Estimado       Error abs.     Error est.  
100          6.624227       0.307663       0.320544    
1000         6.369560       0.052996       0.103837    
10000        6.293961       0.022603       0.032677    
100000       6.312443       0.004121       0.010408    
1000000      6.316263       0.000301       0.003295    

4. ∫_0^1 (1-x^2)^{3/2} dx    (valor exacto = 0.589049)
N            Estimado       Error abs.     Error est.  
100          0.546212       0.042836       0.032225    
1000         0.581614       0.007435       0.010407    
10000        0.590294       0.001245       0.003307    
100000       0.589258       0.000209       0.001049    
1000000      0.589085       0.000037       0.000332    

5. ∫_{-2}^2 e^{x+x^2} dx    (valor exacto = 93.162753)
N            Estimado       Error abs.     Error est.  
100          88.901472      4.261281       21.913040   
1000         94.847830      1.685077       7.865355    

### Ejercicio 8: integral doble sobre el cuadrado unitario

$$\int_0^1\!\!\int_0^1 e^{(x+y)^2}\,dy\,dx \approx \frac{1}{N}\sum_{i=1}^{N} e^{(u_i+v_i)^2}$$

Como la región es $[0,1]\times[0,1]$ (área 1), la integral es directamente el promedio de
$e^{(x+y)^2}$ con $x,y\sim U(0,1)$ independientes. Valor exacto $\approx 4.899159$.

In [ ]:
exacto8 = 4.899159
print("8. ∫_0^1 ∫_0^1 e^{(x+y)^2} dy dx   (valor exacto = %.6f)" % exacto8)
print("%-12s %-14s %-14s" % ("N", "Estimado", "Error abs."))
for N in Ns:
    _, _, U1 = genranN(A_MC, C_MC, M_MC, 11111, N)
    _, _, U2 = genranN(A_MC, C_MC, M_MC, 22222, N)
    u, v = np.array(U1), np.array(U2)
    est = np.mean(np.exp((u + v)**2))
    print("%-12d %-14.6f %-14.6f" % (N, est, abs(est - exacto8)))

8. ∫_0^1 ∫_0^1 e^{(x+y)^2} dy dx   (valor exacto = 4.899159)
N            Estimado       Error abs.    
100          4.071028       0.828131      
1000         4.668887       0.230272      
10000        4.832847       0.066312      
100000       4.909325       0.010166      
1000000      4.908658       0.009499      


### Ejercicio 9: integral doble con la sugerencia de la guía

$$\int_0^{\infty}\!\!\int_0^{x} e^{-(x+y)}\,dy\,dx$$

Con la función indicadora sugerida $I_y(x)=1$ si $y<x$ y $0$ si $y\ge x$, el límite interno se
extiende a infinito:

$$\int_0^{\infty}\!\!\int_0^{\infty} e^{-(x+y)}\,I_y(x)\,dy\,dx$$

y aplicando el cambio $x=\tfrac{1}{u}-1$, $y=\tfrac{1}{v}-1$ en ambas variables se obtiene un
estimador Monte Carlo. Valor exacto $=1/2$.

In [ ]:
exacto9 = 0.5
print("9. ∫_0^inf ∫_0^x e^{-(x+y)} dy dx   (valor exacto = %.6f)" % exacto9)
print("%-12s %-14s %-14s" % ("N", "Estimado", "Error abs."))
for N in Ns:
    _, _, U1 = genranN(A_MC, C_MC, M_MC, 33333, N)
    _, _, U2 = genranN(A_MC, C_MC, M_MC, 44444, N)
    u, v = np.array(U1), np.array(U2)
    x = 1.0/u - 1.0
    y = 1.0/v - 1.0
    indic = (y < x).astype(float)                 # I_y(x)
    muestras = np.exp(-(x + y)) * indic / (u**2 * v**2)
    est = np.mean(muestras)
    print("%-12d %-14.6f %-14.6f" % (N, est, abs(est - exacto9)))

9. ∫_0^inf ∫_0^x e^{-(x+y)} dy dx   (valor exacto = 0.500000)
N            Estimado       Error abs.    
100          0.507368       0.007368      
1000         0.494095       0.005905      
10000        0.489846       0.010154      
100000       0.500733       0.000733      
1000000      0.498913       0.001087      


# 5.3. Generación de variables aleatorias discretas

Se usa el método de la transformada inversa para una v.a. discreta
con $P\{X=x_j\}=p_j$ se genera $U\sim U(0,1)$ y se toma

$$X=x_j \quad\text{si}\quad \sum_{i=1}^{j-1}p_i \le U < \sum_{i=1}^{j}p_i$$


In [ ]:
def genvardiscret(U, X, P):
    V = []
    for t in range(0, len(U)):
        for t2 in range(0, len(X)):
            if U[t] < P[t2]:
                V.append(X[t2])
                break
    return V

X0_53, A_53, C_53, M_53 = 2391, 25214903917, 11, 2**48

In [ ]:

_, _, U100 = genranN(A_53, C_53, M_53, X0_53, 100)
print("Primeros 10 U_i:", np.round(U100[:10], 4))

Primeros 10 U_i: [0.2142 0.745  0.661  0.7595 0.7835 0.3924 0.6607 0.8984 0.3532 0.7253]


a. p_1=0.20, p_2=0.15, p_3=0.25, p_4=0.40

In [ ]:
Xa = [1, 2, 3, 4]
pa = [0.20, 0.15, 0.25, 0.40]
Pa = np.cumsum(pa)
print("p(X) =", pa)
print("P(X) =", np.round(Pa, 4))

Va = genvardiscret(U100, Xa, Pa)
print("\nPrimeros 100 valores X_i:")
print(Va)

props = [Va.count(k)/len(Va) for k in Xa]
print("\nProporción empírica :", [round(x, 3) for x in props])
print("Probabilidad teórica:", pa)

p(X) = [0.2, 0.15, 0.25, 0.4]
P(X) = [0.2  0.35 0.6  1.  ]

Primeros 100 valores X_i:
[2, 4, 4, 4, 4, 3, 4, 4, 3, 4, 2, 1, 4, 4, 1, 1, 1, 1, 2, 3, 4, 3, 1, 3, 2, 4, 3, 4, 1, 3, 4, 2, 4, 2, 1, 4, 4, 1, 4, 4, 4, 1, 2, 1, 4, 3, 2, 3, 4, 2, 4, 3, 3, 3, 4, 4, 4, 2, 4, 2, 1, 4, 3, 3, 3, 3, 4, 1, 4, 4, 1, 3, 1, 4, 4, 3, 2, 1, 4, 4, 2, 1, 3, 1, 4, 3, 3, 3, 4, 3, 3, 1, 1, 3, 2, 4, 3, 4, 1, 3]

Proporción empírica : [0.21, 0.14, 0.27, 0.38]
Probabilidad teórica: [0.2, 0.15, 0.25, 0.4]


b. P\{X=1\}=0.3, P\{X=2\}=0.2, P\{X=3\}=0.35, P\{X=4\}=0.15

In [ ]:
Xb = [1, 2, 3, 4]
pb = [0.30, 0.20, 0.35, 0.15]
Pb = np.cumsum(pb)
print("p(X) =", pb)
print("P(X) =", np.round(Pb, 4))

Vb = genvardiscret(U100, Xb, Pb)
print("\nPrimeros 100 valores X_i:")
print(Vb)

props_b = [Vb.count(k)/len(Vb) for k in Xb]
print("\nProporción empírica :", [round(x, 3) for x in props_b])
print("Probabilidad teórica:", pb)

p(X) = [0.3, 0.2, 0.35, 0.15]
P(X) = [0.3  0.5  0.85 1.  ]

Primeros 100 valores X_i:
[1, 3, 3, 3, 3, 2, 3, 4, 2, 3, 2, 1, 3, 3, 1, 1, 1, 1, 1, 2, 3, 2, 1, 2, 1, 4, 2, 4, 1, 3, 3, 2, 4, 1, 1, 3, 4, 1, 3, 3, 4, 1, 1, 1, 4, 2, 2, 2, 3, 1, 4, 2, 2, 2, 3, 4, 4, 1, 3, 1, 1, 3, 3, 2, 2, 2, 3, 1, 4, 4, 1, 2, 1, 4, 4, 3, 1, 1, 3, 3, 1, 1, 2, 1, 4, 2, 2, 3, 4, 3, 3, 1, 1, 3, 1, 3, 2, 4, 1, 3]

Proporción empírica : [0.32, 0.22, 0.29, 0.17]
Probabilidad teórica: [0.3, 0.2, 0.35, 0.15]


c. p_1=1/3, p_2=2/3 — proporción de valores iguales a 1

Se genera con la misma función `genvardiscret` para $n=100,\ 1000,\ 10000$ y se reporta la
proporción de unos, que debe aproximarse a $p_1=1/3\approx 0.3333$.

In [ ]:
Xc = [1, 2]
pc = [1/3, 2/3]
Pc = np.cumsum(pc)

print("%-8s %-14s %-14s" % ("n", "Prop. de 1s", "Esperado (1/3)"))
for n in [100, 1000, 10000]:
    _, _, Un = genranN(A_53, C_53, M_53, X0_53, n)
    Vc = genvardiscret(Un, Xc, Pc)
    prop1 = Vc.count(1)/n
    print("%-8d %-14.4f %-14.4f" % (n, prop1, 1/3))

n        Prop. de 1s    Esperado (1/3)
100      0.3400         0.3333        
1000     0.3280         0.3333        
10000    0.3316         0.3333        


## d. Método de composición

$X$ toma los valores $1,\dots,10$ con probabilidades $0.06,0.06,0.06,0.06,0.06,0.15,0.13,0.14,0.15,0.13$.
La distribución se escribe como una mezcla de dos componentes:

$$p(x) = 0.30\,p_1(x) + 0.70\,p_2(x)$$

- Componente 1 (peso $0.30$): valores $\{1,2,3,4,5\}$, cada uno con $0.06$. Condicionalmente son
  uniformes ($0.06/0.30 = 0.2$ cada uno).
- Componente 2 (peso $0.70$): valores $\{6,7,8,9,10\}$ con $0.15,0.13,0.14,0.15,0.13$, que
  renormalizadas ($/0.70$) dan la distribución condicional.

Algoritmo: generar $U_1$ para elegir la componente ($U_1<0.30\Rightarrow$ comp. 1); generar
$U_2$ y aplicar transformada inversa dentro de la componente elegida. Se usan dos secuencias
pseudoaleatorias distintas.

In [ ]:
def gen_composicion(U1, U2):
    w1 = 0.30
    cond1 = np.cumsum([0.2, 0.2, 0.2, 0.2, 0.2])
    cond2 = np.cumsum(np.array([0.15, 0.13, 0.14, 0.15, 0.13]) / 0.70)
    V = []
    for u1, u2 in zip(U1, U2):
        if u1 < w1:
            for j in range(5):
                if u2 < cond1[j]:
                    V.append(1 + j)
                    break
        else:
            for j in range(5):
                if u2 < cond2[j]:
                    V.append(6 + j)
                    break
    return V


_, _, U1 = genranN(A_53, C_53, M_53, X0_53, 100)
_, _, U2 = genranN(A_53, C_53, M_53, 987654321, 100)

Vd = gen_composicion(U1, U2)
print("Primeros 100 valores X_i:")
print(Vd)

Xd = list(range(1, 11))
pd_teo = [0.06, 0.06, 0.06, 0.06, 0.06, 0.15, 0.13, 0.14, 0.15, 0.13]
props_d = [Vd.count(k)/len(Vd) for k in Xd]
print("\nProporción empírica :", [round(x, 3) for x in props_d])
print("Probabilidad teórica:", pd_teo)

Primeros 100 valores X_i:
[2, 10, 8, 8, 9, 8, 9, 9, 9, 10, 9, 4, 9, 6, 2, 5, 5, 1, 4, 7, 7, 9, 2, 6, 3, 8, 8, 9, 2, 8, 8, 10, 9, 1, 5, 6, 9, 4, 6, 9, 7, 1, 3, 1, 10, 10, 10, 9, 9, 2, 7, 9, 6, 6, 8, 8, 9, 4, 7, 4, 4, 9, 6, 6, 8, 10, 9, 4, 9, 7, 1, 10, 2, 10, 8, 9, 1, 5, 10, 6, 3, 4, 7, 4, 8, 6, 6, 8, 9, 6, 7, 5, 1, 6, 3, 9, 6, 10, 1, 9]

Proporción empírica : [0.08, 0.06, 0.04, 0.09, 0.05, 0.14, 0.08, 0.13, 0.22, 0.11]
Probabilidad teórica: [0.06, 0.06, 0.06, 0.06, 0.06, 0.15, 0.13, 0.14, 0.15, 0.13]


# 5.4. Simulación ad hoc con generación de variables aleatorias discretas

Repetir lo realizado en el Laboratorio 1 Simulación Ad Hoc para las secciones 5.1. y 5.2. pero generando los
tiempos entre llegadas (Time Between Arrivals) usando su implementación de un generador de variables
aleatorias de Poisson con lambda = 10 y tiempos de servicio (Service Time) usando su implementación de un
generador de variables aleatorias Binomiales con n = 10 y p=0.40

- En el Laboratorio 1 los tiempos entre llegadas eran uniforme discreta en $[1,10]$ y los tiempos de servicio uniforme discreta en $[1,6]$ (dados/ruletas).
- Aquí los tiempos entre llegadas (TEL) se generan con una v.a. de Poisson con $\lambda=10$
  y los tiempos de servicio (TS) con una v.a. Binomial con $n=10$, $p=0.40$,

In [ ]:
def genpoisson(U, L):
    i = 0
    px = np.exp(-L)
    Fx = px
    while True:
        if U < Fx:
            return i
        px = (L*px)/(i+1)
        Fx = Fx + px
        i = i + 1

def genpoissonN(Ui, L):
    return [genpoisson(u, L) for u in Ui]

def genbinomial(U, n, p):
    c = p/(1-p)
    i = 0
    px = np.power(1-p, n)
    Fx = px
    while True:
        if U < Fx:
            return i
        px = (c*(n-i)/(i+1))*px
        Fx = Fx + px
        i = i + 1

def genbinomialN(Ui, n, p):
    return [genbinomial(u, n, p) for u in Ui]

## 5.4.1 (= Lab 1, 5.1) Generación de la tabla ad hoc de la cola

Se construye la Tabla 1.1 de [Banks1998] con las mismas 9 columnas del Laboratorio 1. El cliente 1
llega en el tiempo 0 (no tiene tiempo entre llegadas).

In [ ]:
def simular_adhoc(TEL, TS):

    n = len(TS)
    arrival = [0]*n
    begins  = [0]*n
    ends    = [0]*n
    tsys    = [0]*n
    idle    = [0]*n
    queue   = [0]*n
    for i in range(n):
        prev_end = ends[i-1] if i > 0 else 0
        arrival[i] = 0 if i == 0 else arrival[i-1] + TEL[i]
        begins[i]  = max(arrival[i], prev_end)
        ends[i]    = begins[i] + TS[i]
        tsys[i]    = ends[i] - arrival[i]
        idle[i]    = max(arrival[i] - prev_end, 0)
        queue[i]   = max(prev_end - arrival[i], 0)
    return arrival, begins, ends, tsys, idle, queue

In [ ]:
n_clientes = 20
lam_poisson = 10
n_binom, p_binom = 10, 0.40


_, _, U_lleg = genranN(A_53, C_53, M_53, X0_53, n_clientes)
_, _, U_serv = genranN(A_53, C_53, M_53, 999983, n_clientes)

TEL = genpoissonN(U_lleg, lam_poisson)        # tiempos entre llegadas ~ Poisson(10)
TS  = genbinomialN(U_serv, n_binom, p_binom)  # tiempos de servicio   ~ Binomial(10, 0.4)

arrival, begins, ends, tsys, idle, queue = simular_adhoc(TEL, TS)

print("%-4s %-8s %-8s %-8s %-8s %-8s %-8s %-8s %-8s" %
      ("Cli", "TEL", "Arrival", "TS", "SvcBeg", "SvcEnd", "InSys", "Idle", "InQueue"))
for i in range(n_clientes):
    tel = "-" if i == 0 else TEL[i]
    print("%-4d %-8s %-8d %-8d %-8d %-8d %-8d %-8d %-8d" %
          (i+1, str(tel), arrival[i], TS[i], begins[i], ends[i], tsys[i], idle[i], queue[i]))

print("\nSumas:  Time in System =", sum(tsys),
      "  Idle Time =", sum(idle), "  Time in Queue =", sum(queue))

Cli  TEL      Arrival  TS       SvcBeg   SvcEnd   InSys    Idle     InQueue 
1    -        0        4        0        4        4        0        0       
2    12       12       3        12       15       3        8        0       
3    11       23       4        23       27       4        8        0       
4    12       35       3        35       38       3        8        0       
5    12       47       5        47       52       5        9        0       
6    9        56       5        56       61       5        4        0       
7    11       67       3        67       70       3        6        0       
8    14       81       2        81       83       2        11       0       
9    9        90       3        90       93       3        7        0       
10   12       102      3        102      105      3        9        0       
11   8        110      6        110      116      6        5        0       
12   4        114      2        116      118      4        0        2       

## 5.4.2  Medidas de desempeño

Las cinco medidas del Capítulo 1 de [Banks1998], con las mismas fórmulas del Laboratorio 1

In [ ]:
n = n_clientes
total_run = ends[-1]
n_waited = sum(1 for q in queue if q > 0)

avg_time_system   = sum(tsys) / n
percent_idle      = sum(idle) / total_run * 100
avg_wait_customer = sum(queue) / n
fraction_wait     = n_waited / n
avg_wait_waited   = (sum(queue) / n_waited) if n_waited > 0 else 0.0

print("Media TEL simulada = %.3f  (teórica E[Poisson]=λ=%d)"  % (np.mean(TEL[1:]), lam_poisson))
print("Media TS  simulada = %.3f  (teórica E[Binomial]=np=%.1f)\n" % (np.mean(TS), n_binom*p_binom))

print("Average time in system              = %.3f" % avg_time_system)
print("Percent idle time                   = %.2f %%" % percent_idle)
print("Average waiting time per customer   = %.3f" % avg_wait_customer)
print("Fraction having to wait             = %.3f" % fraction_wait)
print("Average waiting time of those who waited = %.3f" % avg_wait_waited)

Media TEL simulada = 9.474  (teórica E[Poisson]=λ=10)
Media TS  simulada = 3.700  (teórica E[Binomial]=np=4.0)

Average time in system              = 3.800
Percent idle time                   = 59.12 %
Average waiting time per customer   = 0.100
Fraction having to wait             = 0.050
Average waiting time of those who waited = 2.000


## Análisis de resultados

### ¿Qué puede decir de los resultados obtenidos?

De los resultados obtenidos se puede observar que el servidor opera con un alto nivel de desocupación, registrando un 59.12% de tiempo ocioso. Asimismo, el sistema prácticamente no presenta congestión, lo cual se evidencia con un tiempo de espera promedio por cliente de apenas 0.100 y una fracción de clientes que esperan de 0.050, equivalente a solo el 5%. Como resultado de esta baja demanda, el tiempo promedio en el sistema, que es de 3.800, es definido casi en su totalidad por el tiempo de servicio promedio simulado de 3.700, ya que la gran mayoría de los clientes encuentra el servidor disponible inmediatamente al llegar.

### ¿Qué similitudes o diferencias se presentan en esta simulación en comparación con el Laboratorio 1?

En cuanto a las similitudes con el Laboratorio 1, ambas simulaciones conservan las mismas nueve columnas de seguimiento propuestas en la Tabla 1.1 de Banks para registrar el reloj de llegada, el inicio y fin del servicio, y los tiempos de ocio y espera[cite: 1]. Además, se evalúan exactamente las mismas cinco medidas de desempeño en ambos sistemas y los valores generados tanto para las llegadas como para la atención del servidor se mantienen como números enteros[cite: 1]. Por otro lado, las principales diferencias radican en las distribuciones estadísticas, ya que el Laboratorio 1 utilizó exclusivamente distribuciones uniformes discretas para las llegadas y el servicio[cite: 1], mientras que esta nueva simulación emplea distribuciones Poisson y Binomial. Esto altera la utilización teórica del sistema, pues la configuración del Laboratorio 1 generaba llegadas más frecuentes frente a su tiempo de servicio, resultando en una utilización aproximada de 0.64[cite: 1], mientras que en la simulación actual el distanciamiento entre llegadas aumenta y la carga se reduce a una utilización de 0.40. Como consecuencia directa de esta menor carga, el porcentaje de tiempo inactivo en esta simulación se dispara al 59.12%, contrastando fuertemente con el 33.4% promedio obtenido en las corridas de 20 clientes del Laboratorio 1[cite: 1]. De igual forma, la congestión disminuye drásticamente, haciendo que la fracción de personas que deben esperar caiga a un 5%, un valor muy inferior al 31.5% promedio de espera reportado en el primer laboratorio[cite: 1].

# Bibliografía

- [Ross99] Ross, Sheldon. *Simulación*, 2da Edición. Pearson Press, 1999. Capítulos 3 y 4.
- [Mancilla2000] Mancilla Herrera, Alfonso Manuel. *Números aleatorios. Historia, teoría y
  aplicaciones*. Ingeniería y Desarrollo, núm. 8, diciembre, 2000, pp. 49-69. Universidad del
  Norte, Barranquilla, Colombia.
- Cruz Roa, Angel Alfonso. Notebooks del curso Simulación Computacional: `Integración Monte
  Carlo.ipynb`, `Generación de Variables Aleatorias Discretas.ipynb`. Universidad de los Llanos.